# GraphGrasp — Training on Google Colab

**Model:** GCN_CAM_8_8_16_16_32  
**Dataset:** HOGraspNet — 17 classes, taxonomy_v1 collapse (Run 007)

**Setup checklist:**
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- CSV uploaded to Drive:

```
MyDrive/AIST-hand/data/raw/
  hograspnet.csv
```

- GitHub Personal Access Token (repo scope) ready to paste in step 2.

> **Run 007:** Trains on the 17-class data-driven taxonomy derived from Run 006 confusion
> analysis (gcn_thresh=0.15, union-find, experiments/taxonomy_v1).  
> Processed graph cache suffix: `hograspnet_{split}_c17_cmc.pt`.  
> Delete any existing `c28` cache before running. `c17` will be built automatically (~20-30 min).

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Adjust the paths if your files live somewhere else in Drive.

In [ ]:
import os

DRIVE_DATA_DIR   = '/content/drive/MyDrive/AIST-hand/data/raw'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AIST-hand/experiments'
GITHUB_TOKEN     = ''  # paste your token here
GITHUB_USER      = 'isapedraza'
REPO_NAME        = 'AIST-hand'

fname = 'hograspnet.csv'
path = os.path.join(DRIVE_DATA_DIR, fname)
exists = os.path.exists(path)
size_mb = os.path.getsize(path) / 1e6 if exists else 0
print(f"  {'OK' if exists else 'MISSING':8s} {fname} ({size_mb:.0f} MB)")
if not exists:
    raise FileNotFoundError(f'{fname} not found in {DRIVE_DATA_DIR}')

## 3. Clone the repository

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
    print('Already cloned, pulled latest.')

## 4. Install dependencies

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

!pip install -q torch-geometric
!pip install -q scikit-learn tensorboard pandas numpy
!pip install -q -e /content/AIST-hand/grasp-model

print('Done.')

## 5. Link data from Drive

Symlinks to avoid copying the CSVs (~1 GB) into the Colab disk.

In [ ]:
raw_dir       = f'{REPO_DIR}/grasp-model/data/raw'
processed_dir = f'{REPO_DIR}/grasp-model/data/processed'
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

src = os.path.join(DRIVE_DATA_DIR, 'hograspnet.csv')
dst = os.path.join(raw_dir, 'hograspnet.csv')
if not os.path.exists(dst):
    os.symlink(src, dst)
print(f'  {dst}')

## 5b. Restore processed graph cache from Drive (optional)

If you already ran this before, restore the `.pt` files from Drive to skip the 20-30 min graph conversion.

In [ ]:
DRIVE_PROCESSED = '/content/drive/MyDrive/AIST-hand/data/processed'

if os.path.exists(DRIVE_PROCESSED):
    restored = 0
    for fname in os.listdir(DRIVE_PROCESSED):
        src = os.path.join(DRIVE_PROCESSED, fname)
        dst = os.path.join(processed_dir, fname)
        if not os.path.exists(dst):
            import shutil
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1e6
            print(f'  Restored: {fname} ({size_mb:.0f} MB)')
            restored += 1
    if restored == 0:
        print('  Nothing new to restore (already up to date or Drive folder empty).')
else:
    print('  No cache on Drive yet -- will be generated during training (step 6).')

## 6. Train

The first run converts the CSVs to PyG graphs and caches them as `.pt` files (~20-30 min for 1.4 M samples). After that, runs start from the cache.

In [ ]:
os.chdir(f'{REPO_DIR}/grasp-model')
print(f'cwd: {os.getcwd()}')

In [ ]:
!python train.py

## 7. Save model and TensorBoard logs to Drive

In [ ]:
import shutil

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

src = f'{REPO_DIR}/grasp-model/experiments/best_model.pth'
dst = os.path.join(DRIVE_OUTPUT_DIR, 'best_model.pth')
if os.path.exists(src):
    shutil.copy2(src, dst)
    print(f'Saved: {dst}')
else:
    print(f'Not found: {src}')

tb_src = f'{REPO_DIR}/grasp-model/experiments/runs'
tb_dst = os.path.join(DRIVE_OUTPUT_DIR, 'runs')
if os.path.exists(tb_src):
    if os.path.exists(tb_dst):
        shutil.rmtree(tb_dst)
    shutil.copytree(tb_src, tb_dst)
    print(f'TensorBoard logs saved: {tb_dst}')

## 8. TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/AIST-hand/grasp-model/experiments/runs

## 9. Cache processed graphs to Drive (optional)

Save the `.pt` files so the next session skips the conversion step.

In [ ]:
DRIVE_PROCESSED = '/content/drive/MyDrive/AIST-hand/data/processed'
os.makedirs(DRIVE_PROCESSED, exist_ok=True)

local_processed = f'{REPO_DIR}/grasp-model/data/processed'
for fname in os.listdir(local_processed):
    src = os.path.join(local_processed, fname)
    dst = os.path.join(DRIVE_PROCESSED, fname)
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f'  {fname} ({size_mb:.0f} MB)')

## 10. Evaluation -- Plots & Metrics

Generates all analysis artifacts and saves them to Drive. Can be run independently
in a fresh session (needs `best_model.pth` on Drive and the processed cache).

**Outputs saved to** `MyDrive/AIST-hand/experiments/run_007/`:
- `confusion_matrix_norm.png` -- heatmap normalizado por fila con nombres de clase
- `confusion_matrix_norm_gcn.csv` -- matriz numerica (compatible con decide_collapses.py)
- `per_class_f1.png` -- F1 por clase con soporte
- `classification_report.txt` -- precision/recall/F1 por clase
- `results.json` -- metricas resumen

In [ ]:
import os, sys, json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, precision_score, recall_score)
from torch_geometric.loader import DataLoader

# -- Config -------------------------------------------------------------------
REPO_DIR         = '/content/AIST-hand'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/AIST-hand/experiments'
MODEL_PATH       = os.path.join(DRIVE_OUTPUT_DIR, 'best_model.pth')
RUN_DIR          = os.path.join(DRIVE_OUTPUT_DIR, 'run_007')
os.makedirs(RUN_DIR, exist_ok=True)

os.chdir(f'{REPO_DIR}/grasp-model')
if f'{REPO_DIR}/grasp-model/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/grasp-model/src')

from grasp_gcn.dataset.grasps import GraspsClass, TAXONOMY_V1_CLASS_NAMES
from grasp_gcn.network.utils import get_network

NETWORK_TYPE = 'GCN_CAM_8_8_16_16_32'
BATCH_SIZE   = 512
NUM_WORKERS  = 0

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# -- Load test dataset --------------------------------------------------------
dataset_test = GraspsClass(root='data/', split='test', collapse='taxonomy_v1')
test_loader  = DataLoader(dataset_test, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)
NUM_CLASSES  = dataset_test.num_classes
CLASS_NAMES  = [TAXONOMY_V1_CLASS_NAMES[i] for i in range(NUM_CLASSES)]
print(f'Test samples: {len(dataset_test)} | Classes: {NUM_CLASSES}')

# -- Load model ---------------------------------------------------------------
model = get_network(NETWORK_TYPE, dataset_test.num_features, NUM_CLASSES,
                    use_cmc_angle=True).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f'Model loaded from: {MODEL_PATH}')

# -- Inference ----------------------------------------------------------------
y_true, y_pred = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        preds = model(batch).argmax(1).cpu().numpy()
        labels = batch.y.view(-1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

acc      = (y_true == y_pred).mean()
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f'\nTest Accuracy : {acc:.4f}')
print(f'Macro F1      : {macro_f1:.4f}')

# -- 1. Classification report -------------------------------------------------
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print('\n' + report)
report_path = os.path.join(RUN_DIR, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write('Run 007 -- GCN_CAM_8_8_16_16_32 -- taxonomy_v1 (17 classes)\n')
    f.write(f'Test Accuracy : {acc:.4f}\n')
    f.write(f'Macro F1      : {macro_f1:.4f}\n\n')
    f.write(report)
print(f'Saved: {report_path}')

# -- 2. Confusion matrix PNG --------------------------------------------------
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(
    f'Confusion Matrix (row-normalized) -- {NETWORK_TYPE}\n'
    f'Run 007 | taxonomy_v1 | Test acc: {acc:.3f} | Macro F1: {macro_f1:.3f}',
    fontsize=11)
ax.set_xlabel('Predicted', fontsize=10)
ax.set_ylabel('True', fontsize=10)
ticks = np.arange(NUM_CLASSES)
ax.set_xticks(ticks)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
ax.set_yticks(ticks)
ax.set_yticklabels(CLASS_NAMES, fontsize=7)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        val = cm_norm[i, j]
        if val > 0.05:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=5, color='white' if val > 0.6 else 'black')
plt.tight_layout()
cm_png = os.path.join(RUN_DIR, 'confusion_matrix_norm.png')
fig.savefig(cm_png, dpi=150)
plt.show()
print(f'Saved: {cm_png}')

# -- 3. Confusion matrix CSV --------------------------------------------------
cm_df = pd.DataFrame(cm_norm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_csv = os.path.join(RUN_DIR, 'confusion_matrix_norm_gcn.csv')
cm_df.to_csv(cm_csv)
print(f'Saved: {cm_csv}')

# -- 4. Per-class F1 bar chart ------------------------------------------------
f1_per_class   = f1_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
prec_per_class = precision_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
rec_per_class  = recall_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
support        = np.bincount(y_true, minlength=NUM_CLASSES)

colors = ['#d62728' if f < 0.4 else '#ff7f0e' if f < 0.6 else '#2ca02c'
          for f in f1_per_class]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, f1_per_class, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(macro_f1, color='steelblue', linestyle='--', linewidth=1.2,
           label=f'Macro F1 = {macro_f1:.3f}')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('F1 score', fontsize=11)
ax.set_title(f'Per-class F1 -- Run 007 | taxonomy_v1 | {NETWORK_TYPE}', fontsize=12)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
for bar, sup in zip(bars, support):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{sup//1000}k', ha='center', va='bottom', fontsize=6, rotation=90)
plt.tight_layout()
f1_png = os.path.join(RUN_DIR, 'per_class_f1.png')
fig.savefig(f1_png, dpi=150)
plt.show()
print(f'Saved: {f1_png}')

# -- 5. results.json ----------------------------------------------------------
results = {
    'run': 'Run_007',
    'model': NETWORK_TYPE,
    'taxonomy': 'taxonomy_v1',
    'num_classes': NUM_CLASSES,
    'test_accuracy': round(float(acc), 4),
    'macro_f1': round(float(macro_f1), 4),
    'per_class': {
        CLASS_NAMES[i]: {
            'precision': round(float(prec_per_class[i]), 4),
            'recall':    round(float(rec_per_class[i]),  4),
            'f1':        round(float(f1_per_class[i]),   4),
            'support':   int(support[i]),
        }
        for i in range(NUM_CLASSES)
    },
}
results_path = os.path.join(RUN_DIR, 'results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved: {results_path}')

print(f'\nAll artifacts saved to: {RUN_DIR}')